In [1]:
import pandas as pd

df = pd.read_csv("retail_store_sales.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [3]:
df.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


In [4]:
price_recovery = df[
    df["Price Per Unit"].isna() &
    df["Quantity"].notna() &
    df["Total Spent"].notna()
]

print("Recoverable Price rows:", len(price_recovery))

Recoverable Price rows: 609


In [5]:
df.loc[df["Price Per Unit"].isna(), "Price Per Unit"] = (
    df["Total Spent"] / df["Quantity"]
)

In [6]:
df["Price Per Unit"].isnull().sum()

np.int64(0)

In [7]:
quantity_recovery = df[
    df["Quantity"].isna() &
    df["Price Per Unit"].notna() &
    df["Total Spent"].notna()
]

print("Recoverable Quantity rows:", len(quantity_recovery))

Recoverable Quantity rows: 0


In [8]:
total_recovery = df[
    df["Total Spent"].isna() &
    df["Price Per Unit"].notna() &
    df["Quantity"].notna()
]

print("Recoverable Total Spent rows:", len(total_recovery))

Recoverable Total Spent rows: 0


In [9]:
df[df["Item"].isna()]["Category"].value_counts()

,count
Category,
Food,162
Computers and electric accessories,161
Patisserie,159
Milk Products,159
Electric household essentials,154
Butchers,147
Beverages,140
Furniture,131


In [10]:
item_check = df.dropna(
    subset=["Item", "Price Per Unit"]
)

item_check.groupby(
    ["Category", "Price Per Unit"]
)["Item"].nunique().max()


1

In [11]:
item_recovery = df[
    df["Item"].isna() &
    df["Price Per Unit"].notna()
]

print("Recoverable Item rows:", len(item_recovery))

Recoverable Item rows: 1213


In [12]:
item_map = (
    df.dropna(subset=["Item", "Price Per Unit"])
      .drop_duplicates(["Category", "Price Per Unit"])
      .set_index(["Category", "Price Per Unit"])["Item"]
)

In [13]:
df["Item"] = df["Item"].fillna(
    df.apply(
        lambda row: item_map.get(
            (row["Category"], row["Price Per Unit"])
        ),
        axis=1
    )
)

In [14]:
print("Missing Item values:", df["Item"].isnull().sum())

Missing Item values: 0


In [15]:
df["Discount Applied"].value_counts(dropna=False)

,count
Discount Applied,
True,4219
NaN,4199
False,4157


In [16]:
df["Discount Applied"] = df["Discount Applied"].fillna("Unknown")

In [17]:
incomplete_rows = df[
    df["Quantity"].isna() &
    df["Total Spent"].isna()
]

print("Incomplete transaction rows:", len(incomplete_rows))

Incomplete transaction rows: 604


In [18]:
df_clean = df.dropna(
    subset=["Quantity", "Total Spent"]
).copy()

print("Cleaned Dataset Shape:", df_clean.shape)

Cleaned Dataset Shape: (11971, 11)


In [19]:
df_clean.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,0
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


In [20]:
print("Duplicate rows:", df_clean.duplicated().sum())

Duplicate rows: 0


In [21]:
df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"]
)

print(df_clean["Transaction Date"].dtype)

datetime64[ns]


In [22]:
calculated_total = (
    df_clean["Price Per Unit"] * df_clean["Quantity"]
)

difference = (
    df_clean["Total Spent"] - calculated_total
).abs()

print("Maximum difference:", difference.max())

Maximum difference: 0.0


In [23]:
print(df_clean.dtypes)

Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied            object
dtype: object


In [24]:
df_clean.to_csv(
    "retail_store_sales_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [25]:
import pandas as pd

df_clean = pd.read_csv("retail_store_sales_cleaned.csv")

print("Dataset Shape:", df_clean.shape)
df_clean.head()

Dataset Shape: (11971, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [26]:
category_groups = df_clean.groupby("Category")

print(category_groups)

In [27]:
category_sales = df_clean.groupby("Category")["Total Spent"].sum()

category_sales

,Total Spent
Category,
Beverages,197047.5
Butchers,208118.0
Computers and electric accessories,190692.5
Electric household essentials,203813.5
Food,194812.0
Furniture,195310.0
Milk Products,180112.0
Patisserie,182165.5


In [28]:
category_avg_sales = df_clean.groupby("Category")["Total Spent"].mean()

category_avg_sales

,Total Spent
Category,
Beverages,131.716243
Butchers,139.116310
Computers and electric accessories,129.107989
Electric household essentials,134.441623
Food,129.271400
Furniture,128.072131
Milk Products,119.042961
Patisserie,126.416031


In [29]:
category_count = df_clean.groupby("Category")["Transaction ID"].count()

category_count

,Transaction ID
Category,
Beverages,1496
Butchers,1496
Computers and electric accessories,1477
Electric household essentials,1516
Food,1507
Furniture,1525
Milk Products,1513
Patisserie,1441


In [30]:
payment_count = df_clean.groupby("Payment Method")["Transaction ID"].count()

payment_count

,Transaction ID
Payment Method,
Cash,4103
Credit Card,3927
Digital Wallet,3941


In [31]:
location_sales = df_clean.groupby("Location")["Total Spent"].sum()

location_sales

,Total Spent
Location,
In-store,760670.0
Online,791401.0


In [32]:
category_summary = df_clean.groupby("Category")["Total Spent"].agg(
    ["sum", "mean", "min", "max", "count"]
)

category_summary

,sum,mean,min,max,count
Category,,,,,
Beverages,197047.5,131.716243,5.0,410.0,1496
Butchers,208118.0,139.116310,5.0,410.0,1496
Computers and electric accessories,190692.5,129.107989,5.0,410.0,1477
Electric household essentials,203813.5,134.441623,5.0,410.0,1516
Food,194812.0,129.271400,5.0,410.0,1507
Furniture,195310.0,128.072131,5.0,410.0,1525
Milk Products,180112.0,119.042961,5.0,410.0,1513
Patisserie,182165.5,126.416031,5.0,410.0,1441


In [33]:
category_details = df_clean.groupby("Category").agg({
    "Total Spent": ["sum", "mean"],
    "Quantity": ["sum", "mean"]
})

category_details

Total Spent             Quantity          
                                           sum        mean      sum      mean
Category                                                                     
Beverages                             197047.5  131.716243   8358.0  5.586898
Butchers                              208118.0  139.116310   8206.0  5.485294
Computers and electric accessories    190692.5  129.107989   8272.0  5.600542
Electric household essentials         203813.5  134.441623   8309.0  5.480871
Food                                  194812.0  129.271400   8387.0  5.565362
Furniture                             195310.0  128.072131   8462.0  5.548852
Milk Products                         180112.0  119.042961   8339.0  5.511566
Patisserie                            182165.5  126.416031   7943.0  5.512144

In [34]:
customer_spending = (
    df_clean.groupby("Customer ID")["Total Spent"]
    .sum()
)

customer_spending.head(10)

,Total Spent
Customer ID,
CUST_01,58731.5
CUST_02,62046.5
CUST_03,60811.0
CUST_04,61767.5
CUST_05,66974.5
CUST_06,58632.5
CUST_07,60694.5
CUST_08,67351.5
CUST_09,61423.5


In [35]:
top_customers = (
    customer_spending
    .sort_values(ascending=False)
    .head(10)
)

top_customers

,Total Spent
Customer ID,
CUST_24,68452.0
CUST_08,67351.5
CUST_05,66974.5
CUST_16,65570.5
CUST_13,65037.0
CUST_23,64507.0
CUST_10,63155.5
CUST_15,63117.5
CUST_21,62933.0


In [36]:
category_location_sales = (
    df_clean
    .groupby(["Category", "Location"])["Total Spent"]
    .sum()
)

category_location_sales

Category                            Location
Beverages                           In-store     98026.0
                                    Online       99021.5
Butchers                            In-store    101777.0
                                    Online      106341.0
Computers and electric accessories  In-store     87323.5
                                    Online      103369.0
Electric household essentials       In-store     97778.5
                                    Online      106035.0
Food                                In-store     95926.0
                                    Online       98886.0
Furniture                           In-store     99611.0
                                    Online       95699.0
Milk Products                       In-store     88813.0
                                    Online       91299.0
Patisserie                          In-store     91415.0
                                    Online       90750.5
Name: Total Spent, dtype: float64

In [37]:
category_location_df = category_location_sales.reset_index()

category_location_df

,Category,Location,Total Spent
0,Beverages,In-store,98026.0
1,Beverages,Online,99021.5
2,Butchers,In-store,101777.0
3,Butchers,Online,106341.0
4,Computers and electric accessories,In-store,87323.5
5,Computers and electric accessories,Online,103369.0
6,Electric household essentials,In-store,97778.5
7,Electric household essentials,Online,106035.0
8,Food,In-store,95926.0
9,Food,Online,98886.0


In [38]:
overall_summary = df_clean.agg({
    "Total Spent": ["sum", "mean", "min", "max"],
    "Quantity": ["sum", "mean"]
})

overall_summary

,Total Spent,Quantity
sum,1.552071e+06,66276.00000
mean,1.296526e+02,5.53638
min,5.000000e+00,NaN
max,4.100000e+02,NaN


#Day-37

In [39]:
import pandas as pd

df_clean = pd.read_csv("retail_store_sales_cleaned.csv")

print("Dataset Shape:", df_clean.shape)
df_clean.head()

Dataset Shape: (11971, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [40]:
category_sales = (
    df_clean.groupby("Category")["Total Spent"]
    .sum()
    .reset_index()
)

category_sales

,Category,Total Spent
0,Beverages,197047.5
1,Butchers,208118.0
2,Computers and electric accessories,190692.5
3,Electric household essentials,203813.5
4,Food,194812.0
5,Furniture,195310.0
6,Milk Products,180112.0
7,Patisserie,182165.5


In [41]:
category_quantity = (
    df_clean.groupby("Category")["Quantity"]
    .sum()
    .reset_index()
)

category_quantity

,Category,Quantity
0,Beverages,8358.0
1,Butchers,8206.0
2,Computers and electric accessories,8272.0
3,Electric household essentials,8309.0
4,Food,8387.0
5,Furniture,8462.0
6,Milk Products,8339.0
7,Patisserie,7943.0


In [42]:
category_analysis = pd.merge(
    category_sales,
    category_quantity,
    on="Category"
)

category_analysis

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [43]:
category_analysis_inner = pd.merge(
    category_sales,
    category_quantity,
    on="Category",
    how="inner"
)

category_analysis_inner

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [44]:
category_analysis_left = pd.merge(
    category_sales,
    category_quantity,
    on="Category",
    how="left"
)

category_analysis_left

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [45]:
category_analysis_right = pd.merge(
    category_sales,
    category_quantity,
    on="Category",
    how="right"
)

category_analysis_right

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [46]:
category_sales_filtered = category_sales[
    category_sales["Total Spent"] > 190000
]

category_sales_filtered


,Category,Total Spent
0,Beverages,197047.5
1,Butchers,208118.0
2,Computers and electric accessories,190692.5
3,Electric household essentials,203813.5
4,Food,194812.0
5,Furniture,195310.0


In [47]:
left_join_result = pd.merge(
    category_sales_filtered,
    category_quantity,
    on="Category",
    how="left"
)

left_join_result

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0


In [48]:
right_join_result = pd.merge(
    category_sales_filtered,
    category_quantity,
    on="Category",
    how="right"
)

right_join_result

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,NaN,8339.0
7,Patisserie,NaN,7943.0


In [49]:
outer_join_result = pd.merge(
    category_sales_filtered,
    category_quantity,
    on="Category",
    how="outer"
)

outer_join_result

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,NaN,8339.0
7,Patisserie,NaN,7943.0


In [50]:
category_stats = (
    df_clean.groupby("Category")["Total Spent"]
    .agg(["mean", "max"])
    .reset_index()
)

category_stats

,Category,mean,max
0,Beverages,131.716243,410.0
1,Butchers,139.116310,410.0
2,Computers and electric accessories,129.107989,410.0
3,Electric household essentials,134.441623,410.0
4,Food,129.271400,410.0
5,Furniture,128.072131,410.0
6,Milk Products,119.042961,410.0
7,Patisserie,126.416031,410.0


In [51]:
category_avg = (
    df_clean.groupby("Category")["Total Spent"]
    .mean()
    .reset_index(name="Total Spent")
)

category_max = (
    df_clean.groupby("Category")["Total Spent"]
    .max()
    .reset_index(name="Total Spent")
)

category_avg_max = pd.merge(
    category_avg,
    category_max,
    on="Category"
)

category_avg_max

,Category,Total Spent_x,Total Spent_y
0,Beverages,131.716243,410.0
1,Butchers,139.116310,410.0
2,Computers and electric accessories,129.107989,410.0
3,Electric household essentials,134.441623,410.0
4,Food,129.271400,410.0
5,Furniture,128.072131,410.0
6,Milk Products,119.042961,410.0
7,Patisserie,126.416031,410.0


In [52]:
category_avg_max = pd.merge(
    category_avg,
    category_max,
    on="Category",
    suffixes=("_Average", "_Maximum")
)

category_avg_max

,Category,Total Spent_Average,Total Spent_Maximum
0,Beverages,131.716243,410.0
1,Butchers,139.116310,410.0
2,Computers and electric accessories,129.107989,410.0
3,Electric household essentials,134.441623,410.0
4,Food,129.271400,410.0
5,Furniture,128.072131,410.0
6,Milk Products,119.042961,410.0
7,Patisserie,126.416031,410.0


In [53]:
online_data = df_clean[df_clean["Location"] == "Online"]
instore_data = df_clean[df_clean["Location"] == "In-store"]

combined_data = pd.concat(
    [online_data, instore_data],
    ignore_index=True
)

print("Original Shape:", df_clean.shape)
print("Combined Shape:", combined_data.shape)

Original Shape: (11971, 11)
Combined Shape: (11971, 11)


In [54]:
print("Online Rows:", len(online_data))
print("In-store Rows:", len(instore_data))
print("Combined Rows:", len(combined_data))

combined_data.head()

Online Rows: 6068
In-store Rows: 5903
Combined Rows: 11971


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [55]:
sales_by_category = (
    df_clean.groupby("Category")["Total Spent"]
    .sum()
    .rename("Total Sales")
)

quantity_by_category = (
    df_clean.groupby("Category")["Quantity"]
    .sum()
    .rename("Total Quantity")
)

category_joined = sales_by_category.to_frame().join(
    quantity_by_category
)

category_joined

,Total Sales,Total Quantity
Category,,
Beverages,197047.5,8358.0
Butchers,208118.0,8206.0
Computers and electric accessories,190692.5,8272.0
Electric household essentials,203813.5,8309.0
Food,194812.0,8387.0
Furniture,195310.0,8462.0
Milk Products,180112.0,8339.0
Patisserie,182165.5,7943.0


In [56]:
category_summary = (
    df_clean.groupby("Category")
    .agg(
        Total_Sales=("Total Spent", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Average_Sale=("Total Spent", "mean"),
        Transactions=("Transaction ID", "count")
    )
    .reset_index()
)

category_summary

,Category,Total_Sales,Total_Quantity,Average_Sale,Transactions
0,Beverages,197047.5,8358.0,131.716243,1496
1,Butchers,208118.0,8206.0,139.116310,1496
2,Computers and electric accessories,190692.5,8272.0,129.107989,1477
3,Electric household essentials,203813.5,8309.0,134.441623,1516
4,Food,194812.0,8387.0,129.271400,1507
5,Furniture,195310.0,8462.0,128.072131,1525
6,Milk Products,180112.0,8339.0,119.042961,1513
7,Patisserie,182165.5,7943.0,126.416031,1441


In [57]:
category_summary_sorted = category_summary.sort_values(
    "Total_Sales",
    ascending=False
)

category_summary_sorted

,Category,Total_Sales,Total_Quantity,Average_Sale,Transactions
1,Butchers,208118.0,8206.0,139.116310,1496
3,Electric household essentials,203813.5,8309.0,134.441623,1516
0,Beverages,197047.5,8358.0,131.716243,1496
5,Furniture,195310.0,8462.0,128.072131,1525
4,Food,194812.0,8387.0,129.271400,1507
2,Computers and electric accessories,190692.5,8272.0,129.107989,1477
7,Patisserie,182165.5,7943.0,126.416031,1441
6,Milk Products,180112.0,8339.0,119.042961,1513


In [58]:
category_summary_sorted = category_summary_sorted.reset_index(drop=True)

category_summary_sorted

,Category,Total_Sales,Total_Quantity,Average_Sale,Transactions
0,Butchers,208118.0,8206.0,139.116310,1496
1,Electric household essentials,203813.5,8309.0,134.441623,1516
2,Beverages,197047.5,8358.0,131.716243,1496
3,Furniture,195310.0,8462.0,128.072131,1525
4,Food,194812.0,8387.0,129.271400,1507
5,Computers and electric accessories,190692.5,8272.0,129.107989,1477
6,Patisserie,182165.5,7943.0,126.416031,1441
7,Milk Products,180112.0,8339.0,119.042961,1513


#Day-38

In [59]:
print("Dataset Shape:", df_clean.shape)
df_clean.head()

Dataset Shape: (11971, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [60]:
category_pivot = pd.pivot_table(
    df_clean,
    values="Total Spent",
    index="Category",
    aggfunc="sum"
)

category_pivot

,Total Spent
Category,
Beverages,197047.5
Butchers,208118.0
Computers and electric accessories,190692.5
Electric household essentials,203813.5
Food,194812.0
Furniture,195310.0
Milk Products,180112.0
Patisserie,182165.5


In [61]:
category_location_pivot = pd.pivot_table(
    df_clean,
    values="Total Spent",
    index="Category",
    columns="Location",
    aggfunc="sum"
)

category_location_pivot

Location,In-store,Online
Category,,
Beverages,98026.0,99021.5
Butchers,101777.0,106341.0
Computers and electric accessories,87323.5,103369.0
Electric household essentials,97778.5,106035.0
Food,95926.0,98886.0
Furniture,99611.0,95699.0
Milk Products,88813.0,91299.0
Patisserie,91415.0,90750.5


In [62]:
category_location_stats = pd.pivot_table(
    df_clean,
    values="Total Spent",
    index="Category",
    columns="Location",
    aggfunc=["sum", "mean"]
)

category_location_stats

sum                  mean            
Location                            In-store    Online    In-store      Online
Category                                                                      
Beverages                            98026.0   99021.5  131.755376  131.677527
Butchers                            101777.0  106341.0  136.981157  141.223108
Computers and electric accessories   87323.5  103369.0  126.555797  131.345616
Electric household essentials        97778.5  106035.0  133.213215  135.594629
Food                                 95926.0   98886.0  127.391766  131.148541
Furniture                            99611.0   95699.0  129.029793  127.090305
Milk Products                        88813.0   91299.0  118.417333  119.657929
Patisserie                           91415.0   90750.5  127.496513  125.345994

In [63]:
category_payment_pivot = pd.pivot_table(
    df_clean,
    values="Total Spent",
    index="Category",
    columns="Payment Method",
    aggfunc="sum",
    fill_value=0
)

category_payment_pivot

Payment Method,Cash,Credit Card,Digital Wallet
Category,,,
Beverages,70249.5,61310.5,65487.5
Butchers,72638.0,72832.5,62647.5
Computers and electric accessories,68063.0,63348.0,59281.5
Electric household essentials,68011.0,60514.0,75288.5
Food,69710.0,63540.5,61561.5
Furniture,67565.0,64779.0,62966.0
Milk Products,59882.0,60822.0,59408.0
Patisserie,61591.5,59935.5,60638.5


In [64]:
category_location_total = pd.pivot_table(
    df_clean,
    values="Total Spent",
    index="Category",
    columns="Location",
    aggfunc="sum",
    margins=True
)

category_location_total

Location,In-store,Online,All
Category,,,
Beverages,98026.0,99021.5,197047.5
Butchers,101777.0,106341.0,208118.0
Computers and electric accessories,87323.5,103369.0,190692.5
Electric household essentials,97778.5,106035.0,203813.5
Food,95926.0,98886.0,194812.0
Furniture,99611.0,95699.0,195310.0
Milk Products,88813.0,91299.0,180112.0
Patisserie,91415.0,90750.5,182165.5
All,760670.0,791401.0,1552071.0


In [65]:
category_payment_count = pd.crosstab(
    df_clean["Category"],
    df_clean["Payment Method"]
)

category_payment_count

Payment Method,Cash,Credit Card,Digital Wallet
Category,,,
Beverages,554,457,485
Butchers,495,528,473
Computers and electric accessories,521,490,466
Electric household essentials,518,466,532
Food,528,482,497
Furniture,509,506,510
Milk Products,497,519,497
Patisserie,481,479,481


In [66]:
category_payment_total = pd.crosstab(
    df_clean["Category"],
    df_clean["Payment Method"],
    margins=True
)

category_payment_total

Payment Method,Cash,Credit Card,Digital Wallet,All
Category,,,,
Beverages,554,457,485,1496
Butchers,495,528,473,1496
Computers and electric accessories,521,490,466,1477
Electric household essentials,518,466,532,1516
Food,528,482,497,1507
Furniture,509,506,510,1525
Milk Products,497,519,497,1513
Patisserie,481,479,481,1441
All,4103,3927,3941,11971


In [67]:
final_category_pivot = pd.pivot_table(
    df_clean,
    values=["Total Spent", "Quantity", "Transaction ID"],
    index="Category",
    aggfunc={
        "Total Spent": ["sum", "mean"],
        "Quantity": "sum",
        "Transaction ID": "count"
    }
)

final_category_pivot

Quantity Total Spent            \
                                        sum        mean       sum   
Category                                                            
Beverages                            8358.0  131.716243  197047.5   
Butchers                             8206.0  139.116310  208118.0   
Computers and electric accessories   8272.0  129.107989  190692.5   
Electric household essentials        8309.0  134.441623  203813.5   
Food                                 8387.0  129.271400  194812.0   
Furniture                            8462.0  128.072131  195310.0   
Milk Products                        8339.0  119.042961  180112.0   
Patisserie                           7943.0  126.416031  182165.5   

                                   Transaction ID  
                                            count  
Category                                           
Beverages                                    1496  
Butchers                                     1496  
Computers and electric accessories           1477  
Electric household essentials                1516  
Food                                         1507  
Furniture                                    1525  
Milk Products                                1513  
Patisserie                                   1441

In [68]:
final_category_pivot.sort_values(
    ("Total Spent", "sum"),
    ascending=False
)

Quantity Total Spent            \
                                        sum        mean       sum   
Category                                                            
Butchers                             8206.0  139.116310  208118.0   
Electric household essentials        8309.0  134.441623  203813.5   
Beverages                            8358.0  131.716243  197047.5   
Furniture                            8462.0  128.072131  195310.0   
Food                                 8387.0  129.271400  194812.0   
Computers and electric accessories   8272.0  129.107989  190692.5   
Patisserie                           7943.0  126.416031  182165.5   
Milk Products                        8339.0  119.042961  180112.0   

                                   Transaction ID  
                                            count  
Category                                           
Butchers                                     1496  
Electric household essentials                1516  
Beverages                                    1496  
Furniture                                    1525  
Food                                         1507  
Computers and electric accessories           1477  
Patisserie                                   1441  
Milk Products                                1513

#Day-39

In [69]:
print("Dataset Shape:", df_clean.shape)
print("\nData Type of Transaction Date:")
print(df_clean["Transaction Date"].dtype)

df_clean[["Transaction ID", "Transaction Date", "Total Spent"]].head()

Dataset Shape: (11971, 11)

Data Type of Transaction Date:
object


,Transaction ID,Transaction Date,Total Spent
0,TXN_6867343,2024-04-08,185.0
1,TXN_3731986,2023-07-23,261.0
2,TXN_9303719,2022-10-05,43.0
3,TXN_9458126,2022-05-07,247.5
4,TXN_4575373,2022-10-02,87.5


In [70]:
df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"]
)

print("Data Type After Conversion:")
print(df_clean["Transaction Date"].dtype)

df_clean[["Transaction Date"]].head()

Data Type After Conversion:
datetime64[ns]


,Transaction Date
0,2024-04-08
1,2023-07-23
2,2022-10-05
3,2022-05-07
4,2022-10-02


In [71]:
df_clean["Year"] = df_clean["Transaction Date"].dt.year
df_clean["Month"] = df_clean["Transaction Date"].dt.month
df_clean["Day"] = df_clean["Transaction Date"].dt.day

df_clean[
    ["Transaction Date", "Year", "Month", "Day"]
].head()

,Transaction Date,Year,Month,Day
0,2024-04-08,2024,4,8
1,2023-07-23,2023,7,23
2,2022-10-05,2022,10,5
3,2022-05-07,2022,5,7
4,2022-10-02,2022,10,2


In [72]:
df_clean["Day Name"] = df_clean["Transaction Date"].dt.day_name()

df_clean[
    ["Transaction Date", "Day Name"]
].head()

,Transaction Date,Day Name
0,2024-04-08,Monday
1,2023-07-23,Sunday
2,2022-10-05,Wednesday
3,2022-05-07,Saturday
4,2022-10-02,Sunday


In [73]:
yearly_sales = (
    df_clean.groupby("Year")["Total Spent"]
    .sum()
    .sort_index()
)

yearly_sales

,Total Spent
Year,
2022,510329.5
2023,491312.0
2024,524881.0
2025,25548.5


In [74]:
monthly_sales = (
    df_clean.groupby("Month")["Total Spent"]
    .sum()
    .sort_index()
)

monthly_sales

,Total Spent
Month,
1,174421.0
2,119685.0
3,122392.0
4,125618.5
5,124594.5
6,129771.0
7,131509.0
8,123287.5
9,129344.0


In [75]:
df_clean["Month Name"] = df_clean["Transaction Date"].dt.month_name()

df_clean[
    ["Transaction Date", "Month", "Month Name"]
].head()

,Transaction Date,Month,Month Name
0,2024-04-08,4,April
1,2023-07-23,7,July
2,2022-10-05,10,October
3,2022-05-07,5,May
4,2022-10-02,10,October


In [76]:
day_sales = (
    df_clean.groupby("Day Name")["Total Spent"]
    .sum()
)

day_sales

,Total Spent
Day Name,
Friday,232786.0
Monday,213090.5
Saturday,224191.0
Sunday,225991.0
Thursday,219380.5
Tuesday,219650.0
Wednesday,216982.0


In [77]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

day_sales_ordered = day_sales.reindex(weekday_order)

day_sales_ordered

,Total Spent
Day Name,
Monday,213090.5
Tuesday,219650.0
Wednesday,216982.0
Thursday,219380.5
Friday,232786.0
Saturday,224191.0
Sunday,225991.0


In [78]:
year_month_sales = (
    df_clean.groupby(["Year", "Month"])["Total Spent"]
    .sum()
    .sort_index()
)

year_month_sales

Year  Month
2022  1        52911.5
      2        43325.5
      3        40996.0
      4        40442.0
      5        40347.5
      6        42576.0
      7        44471.5
      8        41333.5
      9        46113.5
      10       38355.0
      11       41256.5
      12       38201.0
2023  1        48052.5
      2        39214.5
      3        38534.5
      4        38905.5
      5        40480.5
      6        42474.0
      7        45632.5
      8        38592.0
      9        41069.0
      10       38322.0
      11       37014.0
      12       43021.0
2024  1        47908.5
      2        37145.0
      3        42861.5
      4        46271.0
      5        43766.5
      6        44721.0
      7        41405.0
      8        43362.0
      9        42161.5
      10       42736.5
      11       44076.0
      12       48466.5
2025  1        25548.5
Name: Total Spent, dtype: float64

In [79]:
start_date = "2024-01-01"
end_date = "2024-12-31"

filtered_data = df_clean[
    (df_clean["Transaction Date"] >= start_date) &
    (df_clean["Transaction Date"] <= end_date)
]

print("Filtered Rows:", len(filtered_data))

filtered_data[
    ["Transaction ID", "Transaction Date", "Total Spent"]
].head()

Filtered Rows: 4036


,Transaction ID,Transaction Date,Total Spent
0,TXN_6867343,2024-04-08,185.0
8,TXN_2722661,2024-03-14,109.5
9,TXN_8776416,2024-12-14,72.0
14,TXN_7563311,2024-11-16,232.0
17,TXN_9939063,2024-01-14,126.0


In [80]:
sales_2024 = filtered_data["Total Spent"].sum()

print("Total Sales in 2024:", sales_2024)

Total Sales in 2024: 524881.0


In [81]:
sorted_data = df_clean.sort_values(
    "Transaction Date"
)

sorted_data[
    ["Transaction ID", "Transaction Date", "Total Spent"]
].head()

,Transaction ID,Transaction Date,Total Spent
10317,TXN_3113686,2022-01-01,350.0
3169,TXN_6950467,2022-01-01,75.0
2932,TXN_7331955,2022-01-01,66.5
5509,TXN_6329331,2022-01-01,77.5
8720,TXN_2751393,2022-01-01,305.0


In [82]:
earliest_date = df_clean["Transaction Date"].min()
latest_date = df_clean["Transaction Date"].max()

print("Earliest Transaction Date:", earliest_date)
print("Latest Transaction Date:", latest_date)

Earliest Transaction Date: 2022-01-01 00:00:00
Latest Transaction Date: 2025-01-18 00:00:00


In [83]:
monthly_trend = (
    df_clean
    .set_index("Transaction Date")
    .resample("ME")["Total Spent"]
    .sum()
)

monthly_trend.head(10)

,Total Spent
Transaction Date,
2022-01-31,52911.5
2022-02-28,43325.5
2022-03-31,40996.0
2022-04-30,40442.0
2022-05-31,40347.5
2022-06-30,42576.0
2022-07-31,44471.5
2022-08-31,41333.5
2022-09-30,46113.5


In [84]:
highest_sales_month = monthly_trend.idxmax()
highest_sales_value = monthly_trend.max()

print("Highest Sales Month:", highest_sales_month)
print("Highest Monthly Sales:", highest_sales_value)

Highest Sales Month: 2022-01-31 00:00:00
Highest Monthly Sales: 52911.5


In [85]:
lowest_sales_month = monthly_trend.idxmin()
lowest_sales_value = monthly_trend.min()

print("Lowest Sales Month:", lowest_sales_month)
print("Lowest Monthly Sales:", lowest_sales_value)

Lowest Sales Month: 2025-01-31 00:00:00
Lowest Monthly Sales: 25548.5


In [86]:
daily_sales = (
    df_clean
    .set_index("Transaction Date")
    .resample("D")["Total Spent"]
    .sum()
)

daily_sales.head(10)

,Total Spent
Transaction Date,
2022-01-01,1642.5
2022-01-02,1135.5
2022-01-03,823.0
2022-01-04,1117.5
2022-01-05,2227.5
2022-01-06,763.5
2022-01-07,1372.5
2022-01-08,1496.5
2022-01-09,1866.0


In [87]:
highest_sales_day = daily_sales.idxmax()
highest_daily_sales = daily_sales.max()

print("Highest Sales Day:", highest_sales_day)
print("Highest Daily Sales:", highest_daily_sales)

Highest Sales Day: 2024-04-09 00:00:00
Highest Daily Sales: 3790.5


In [88]:
lowest_sales_day = daily_sales.idxmin()
lowest_daily_sales = daily_sales.min()

print("Lowest Sales Day:", lowest_sales_day)
print("Lowest Daily Sales:", lowest_daily_sales)

Lowest Sales Day: 2023-05-30 00:00:00
Lowest Daily Sales: 79.0


#DAY-40-Pandas Practical Consolidation...

In [89]:

# Day 40 — Pandas Practical Consolidation

import pandas as pd
import numpy as np

print("Dataset Shape:", df_clean.shape)

print("\nFirst 5 Rows:")
display(df_clean.head())

print("\nColumn Names:")
print(df_clean.columns.tolist())

print("\nMissing Values:")
print(df_clean.isnull().sum().sum())

print("\nData Types:")
print(df_clean.dtypes)

Dataset Shape: (11971, 16)

First 5 Rows:


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Year,Month,Day,Day Name,Month Name
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4,8,Monday,April
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7,23,Sunday,July
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10,5,Wednesday,October
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown,2022,5,7,Saturday,May
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10,2,Sunday,October



Column Names:
['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied', 'Year', 'Month', 'Day', 'Day Name', 'Month Name']

Missing Values:
0

Data Types:
Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied            object
Year                         int32
Month                        int32
Day                          int32
Day Name                    object
Month Name                  object
dtype: object


In [90]:

# Cell 2 — Numerical Summary

numeric_columns = [
    "Price Per Unit",
    "Quantity",
    "Total Spent"
]

display(df_clean[numeric_columns].describe())


,Price Per Unit,Quantity,Total Spent
count,11971.000000,11971.000000,11971.000000
mean,23.360872,5.536380,129.652577
std,10.741889,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [91]:

# Cell 3 — Category-wise Sales Analysis

category_sales = (
    df_clean.groupby("Category")["Total Spent"]
    .agg(["sum", "mean", "count"])
    .round(2)
    .sort_values("sum", ascending=False)
)

category_sales.columns = [
    "Total Sales",
    "Average Sale",
    "Transactions"
]

display(category_sales)


,Total Sales,Average Sale,Transactions
Category,,,
Butchers,208118.0,139.12,1496
Electric household essentials,203813.5,134.44,1516
Beverages,197047.5,131.72,1496
Furniture,195310.0,128.07,1525
Food,194812.0,129.27,1507
Computers and electric accessories,190692.5,129.11,1477
Patisserie,182165.5,126.42,1441
Milk Products,180112.0,119.04,1513


In [92]:

# Cell 4 — Online vs In-store Analysis

location_analysis = (
    df_clean.groupby("Location")["Total Spent"]
    .agg(["sum", "mean", "count"])
    .round(2)
)

location_analysis.columns = [
    "Total Sales",
    "Average Sale",
    "Transactions"
]

display(location_analysis)

print("\nTotal sales by location:")
print(
    location_analysis["Total Sales"]
    .sort_values(ascending=False)
)


,Total Sales,Average Sale,Transactions
Location,,,
In-store,760670.0,128.86,5903
Online,791401.0,130.42,6068



Total sales by location:
Location
Online      791401.0
In-store    760670.0
Name: Total Sales, dtype: float64


In [93]:

# Cell 5 — Payment Method Analysis

payment_analysis = (
    df_clean.groupby("Payment Method")["Total Spent"]
    .agg(["sum", "mean", "count"])
    .round(2)
)

payment_analysis.columns = [
    "Total Sales",
    "Average Sale",
    "Transactions"
]

payment_analysis = payment_analysis.sort_values(
    "Total Sales",
    ascending=False
)

display(payment_analysis)


,Total Sales,Average Sale,Transactions
Payment Method,,,
Cash,537710.0,131.05,4103
Digital Wallet,507279.0,128.72,3941
Credit Card,507082.0,129.13,3927


In [94]:

# Cell 6 — Monthly Sales Analysis

monthly_sales = (
    df_clean.groupby("Month")["Total Spent"]
    .agg(["sum", "mean", "count"])
    .round(2)
)

monthly_sales.columns = [
    "Total Sales",
    "Average Sale",
    "Transactions"
]

display(monthly_sales.sort_values("Total Sales", ascending=False))

print("\nHighest Sales Month:")
print(monthly_sales["Total Sales"].idxmax())

print("\nLowest Sales Month:")
print(monthly_sales["Total Sales"].idxmin())


,Total Sales,Average Sale,Transactions
Month,,,
1,174421.0,134.69,1295
7,131509.0,126.57,1039
6,129771.0,130.95,991
12,129688.5,133.15,974
9,129344.0,131.45,984
4,125618.5,131.81,953
5,124594.5,127.40,978
8,123287.5,124.28,992
3,122392.0,126.83,965



Highest Sales Month:
1

Lowest Sales Month:
10


In [95]:

# Cell 7 — Category vs Location

category_location_pivot = pd.pivot_table(
    df_clean,
    index="Category",
    columns="Location",
    values="Total Spent",
    aggfunc="sum",
    fill_value=0
)

category_location_pivot["Total Sales"] = (
    category_location_pivot.sum(axis=1)
)

category_location_pivot = category_location_pivot.sort_values(
    "Total Sales",
    ascending=False
)

display(category_location_pivot.round(2))


Location,In-store,Online,Total Sales
Category,,,
Butchers,101777.0,106341.0,208118.0
Electric household essentials,97778.5,106035.0,203813.5
Beverages,98026.0,99021.5,197047.5
Furniture,99611.0,95699.0,195310.0
Food,95926.0,98886.0,194812.0
Computers and electric accessories,87323.5,103369.0,190692.5
Patisserie,91415.0,90750.5,182165.5
Milk Products,88813.0,91299.0,180112.0


In [96]:

# Cell 8 — Final Business Summary

total_sales = df_clean["Total Spent"].sum()
total_transactions = df_clean["Transaction ID"].nunique()
total_quantity = df_clean["Quantity"].sum()
average_sale = df_clean["Total Spent"].mean()

top_category = category_sales["Total Sales"].idxmax()
top_category_sales = category_sales["Total Sales"].max()

top_location = location_analysis["Total Sales"].idxmax()
top_location_sales = location_analysis["Total Sales"].max()

print("===== RETAIL STORE SALES SUMMARY =====")
print(f"Total Sales: {total_sales:,.2f}")
print(f"Total Transactions: {total_transactions:,}")
print(f"Total Quantity Sold: {total_quantity:,.0f}")
print(f"Average Transaction Value: {average_sale:,.2f}")
print(f"Top Category: {top_category} ({top_category_sales:,.2f})")
print(f"Top Location: {top_location} ({top_location_sales:,.2f})")


===== RETAIL STORE SALES SUMMARY =====
Total Sales: 1,552,071.00
Total Transactions: 11,971
Total Quantity Sold: 66,276
Average Transaction Value: 129.65
Top Category: Butchers (208,118.00)
Top Location: Online (791,401.00)


In [97]:

# Cell 9 — Final Validation

print("===== FINAL VALIDATION =====")

print("Rows:", len(df_clean))
print("Columns:", df_clean.shape[1])
print("Missing Values:", df_clean.isnull().sum().sum())
print("Duplicate Rows:", df_clean.duplicated().sum())

# Verify transaction arithmetic
calculated_total = (
    df_clean["Price Per Unit"] * df_clean["Quantity"]
)

difference = (
    calculated_total - df_clean["Total Spent"]
).abs().max()

print("Maximum Sales Calculation Difference:", difference)

# Verify location totals against overall sales
print(
    "Location Sales Match:",
    round(location_analysis["Total Sales"].sum(), 2)
    == round(total_sales, 2)
)


===== FINAL VALIDATION =====
Rows: 11971
Columns: 16
Missing Values: 0
Duplicate Rows: 0
Maximum Sales Calculation Difference: 0.0
Location Sales Match: True
